In [ ]:
from all_functionality import *
from config import load_corpora_text

corpora_text = load_corpora_text()


# DEMO

#### TEST

In [ ]:
test = await async_chat_wrapper(
    messages=[
        {
            "role": "user",
            "content": "Hello, how are you?"
        }
    ],
    json_output=True
)
test


{'status': 'good',
 'message': "Hello! As an AI, I don't *feel* in the same way humans do, but my systems are functioning optimally. I am ready to assist you. How can I help you today?"}

In [ ]:
endpoint_logic = """
The Notion API is used to manage databases, data sources, pages, and blocks.

*   **Databases:** `GET /v1/databases/{database_id}` retrieves database information, including its data sources.
*   **Data Sources:** `GET /v1/data_sources/{data_source_id}` retrieves details about a specific data source, including its properties.
*   **Pages:** `POST /v1/pages` creates new pages, optionally setting properties and content. `PATCH /v1/pages/{page_id}` updates existing pages.
*   **Blocks:** `PATCH /v1/blocks/{block_id}` updates block content. `PATCH /v1/blocks/{block_id}/children` appends new blocks as children to a specified block.
"""

endpoint_logic = await async_chat_wrapper(messages=[
    {
        "role": "user",
        "content": corpora_text
    },
    {
        "role": "user",
        "content": "Provide a bullet point summary of how the Notion API endpoints are used in the provided corpora. Focus on the main logic and patterns of usage across the different endpoints. Must explain the exact structure and usage of endpoint url. Limit the summary to 100 words."
    }
], json_output=False, max_tokens=500, temperature=0.5, model_size="small")


In [ ]:
e = openai.embeddings.create(model="gemini-embedding-001", input=corpora_text[:50000]).data[0].embedding


RateLimitError: Error code: 429 - {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

#### RAG DEMO

In [ ]:
### Vector Search Demo

# Assumes storage + corpora_vectorized are already built (from hierarchy demo above)
# Run with top_k=3 to get the three most relevant chunks for a sample query

_demo_query = "How do I add a task with a due date and select property to a Notion database?"

_results = await search_vectors(
    nodes=corpora_vectorized,
    storage=storage,
    query=_demo_query,
    top_k=5,
    threshold=None,
)

print(f"Query: '{_demo_query}'\n")
for i, r in enumerate(_results, 1):
    print(f"[{i}] score={r.score:.4f}  layer={r.layer}")
    print(f"     {r.text[:120].replace(chr(10), ' ')}…")
    print()


Query: 'How do data sources work in Notion'

[1] score=0.7339  layer=1
     src="https://mintcdn.com/notion-demo/S-I3qLQnwRa7HjdK/images/reference/image-3.png?fit=max&auto=format&n=S-I3qLQnwRa7Hjd…

[2] score=0.7297  layer=1
     | `[ { "category": "image", "file": { "url": "https://s3.us-west-2.amazonaws.com/9bc6c6e0-32b8-4d55-8c12-3ae931f43a01/me…

[3] score=0.7267  layer=1
     | `"data_source"`                                                                                                       …

[4] score=0.7210  layer=1
     | | `last_edited_time` | `string` ([ISO 8601 date and time](https://en.wikipedia.org/wiki/ISO_8601))      | Date and tim…

[5] score=0.7152  layer=1
     source](/reference/retrieve-a-data-source) * [Query a data source](/reference/query-a-data-source) ## Object fields <Not…



In [ ]:
summary = await summarize_retrieval_results(_results)
print(summary)


## Notion Data Sources - Summary:

* **What they are:** Data sources are individual tables of data *within* a Notion database. Pages are items *in* a data source.
* **API Access:**
    * Retrieve a list of data sources for a database using the `Retrieve a database` API.
    * Get a data source's `properties` using its `ID` via the `Retrieve a data source` API.
    * Data source IDs can be copied directly from the database settings in the Notion app ("Manage data sources").
* **Key Properties of a Data Source:**
    * **`id`:** Unique identifier (UUID).
    * **`properties`:**  Defines the schema (structure) of the data within the source.
    * **`parent`:** Information about the database the source belongs to.
    * **`last_edited_time/by`:**  Tracking information.
    * **`title`:** Name of the data source.
* **New API Model:** Previously, databases had only one data source, but now can have multiple.
* **Capabilities:** Some properties require "read content" capabilities to access vi

In [ ]:
### QueryEngineer Demo

# Default: temperature=0.3, model_size="gemma27"
qe = QueryEngineer()
_base_query = "How do I add a task with a due date and select property to a Notion database?"

print("── Multi-Query (default temperature=0.3) ────────────────")
for q in await qe.multi_query(_base_query, n=3):
    print(f"  • {q}")

print("\n── CoT Decomposition ────────────────────────────────────")
for q in await qe.cot_decompose(_base_query):
    print(f"  • {q}")

print("\n── Domain Decomposition (Notion) ────────────────────────")
for q in await qe.domain_decompose(_base_query):
    print(f"  • {q}")

# Example: Custom initialization with higher temperature for diversity
# qe_diverse = QueryEngineer(temperature=0.7, model_size="gemma12")


── Multi-Query (default temperature=0.3) ────────────────
  • Notion: create a database entry with a due date and a select option.
  • How can I set a due date and category for tasks in a Notion database?
  • Notion database: adding tasks with deadlines and custom properties (select).

── CoT Decomposition ────────────────────────────────────
  • What are Notion databases and how are they structured?
  • How do you create a new page within a Notion database?
  • What are Notion properties and what types are available?
  • How do you add a 'Date' property to a Notion database?
  • How do you add a 'Select' or 'Multi-select' property to a Notion database?
  • How do you set a due date using the 'Date' property in a Notion database?
  • How do you select an option from the 'Select' or 'Multi-select' property in a Notion database?
  • How do I combine adding a task, setting a due date, and selecting a property value when creating a new database entry in Notion?

── Domain Decomposition (No

In [ ]:
### Multi-Query Search Demo

# Create a QueryEngineer instance (low temperature for focused queries)
qe_search = QueryEngineer(temperature=0.2, model_size="gemma27")

# Base query for decomposition
_base_query = "How do I add a task with a due date and select property to a Notion database?"

# Generate multiple query perspectives
multi_queries = await qe_search.multi_query(_base_query, n=3)
cot_queries = await qe_search.cot_decompose(_base_query)

print(f"Running {len(multi_queries)} multi-queries…")
print(f"Running {len(cot_queries)} chain-of-thought sub-questions…\n")

# Create a partial function binding the search parameters
search_partial = partial(
    search_vectors,
    nodes=corpora_vectorized,
    storage=storage,
    top_k=5,
    threshold=0.6,
)

# Search using multi-queries
print("── Multi-Query Results ──────────────────────────────────────")
consolidated = await search_multiple_queries(multi_queries, search_partial)
print(f"Found {len(consolidated)} unique nodes after deduplication\n")

for i, r in enumerate(consolidated[:5], 1):
    print(f"[{i}] score={r.score:.4f}  layer={r.layer}")
    print(f"     {r.text[:100].replace(chr(10), ' ')}…\n")

# Search using CoT sub-questions
print("\n── CoT Decomposition Results ────────────────────────────────")
consolidated_cot = await search_multiple_queries(cot_queries, search_partial)
print(f"Found {len(consolidated_cot)} unique nodes after deduplication\n")

for i, r in enumerate(consolidated_cot[:5], 1):
    print(f"[{i}] score={r.score:.4f}  layer={r.layer}")
    print(f"     {r.text[:100].replace(chr(10), ' ')}…\n")


Running 3 multi-queries…
Running 8 chain-of-thought sub-questions…

── Multi-Query Results ──────────────────────────────────────
Found 8 unique nodes after deduplication

[1] score=0.7211  layer=1
     'default', 'description': None}, {'id': '9de13799-763a-4e9f-849b-9b4d4d0206be', 'name': 'Tomorrow', …

[2] score=0.6966  layer=1
     ## Tasks data source description and properties {'object': 'data_source', 'id': '0c0c2dea-6c50-4abb-…

[3] score=0.6896  layer=1
     None}, {'id': '{CWD', 'name': 'high', 'color': 'yellow', 'description': None}, {'id': 'Z|L\\', 'name…

[4] score=0.6893  layer=1
     | Name of the database as it appears in Notion. See [rich text object](/reference/rich-text)) for a …

[5] score=0.6770  layer=1
     '1216a3f6-4d84-4f0b-be9f-3f48a14dac7c', 'name': 'To-do', 'color': 'gray', 'option_ids': ['bc969f55-5…


── CoT Decomposition Results ────────────────────────────────
Found 21 unique nodes after deduplication

[1] score=0.7082  layer=1
     Other properties requ

In [ ]:
### RAG Evaluator Demo

evaluator = RagEvaluator(model_size="gemma27", temperature=0.1)

_eval_cases = [
    {
        "label": "Multi-query search",
        "query": _base_query,              # reuse query from the search demo above
        "results": consolidated,           # multi-query deduplicated results
    },
    {
        "label": "CoT search",
        "query": _base_query,
        "results": consolidated_cot,       # CoT deduplicated results
    },
]

for case in _eval_cases:
    grade: RetrievalGrade = evaluator.evaluate(case["query"], case["results"])

    ep_display = {
        EndpointPresence.PRESENT:     "✅ present",
        EndpointPresence.NOT_PRESENT: "❌ not present",
        EndpointPresence.NOT_NEEDED:  "➖ not needed",
    }[grade.endpoint_presence]

    props = (
        f"{grade.properties_discussed} / {grade.properties_total}"
        if grade.properties_total > 0
        else "0 / 0 (N/A)"
    )

    print(f"══ {case['label']} ({len(case['results'])} chunks) ══")
    print(f"  Topic matched      : {'✅' if grade.topic_matched else '❌'}")
    print(f"  Objects coverage   : {grade.objects_coverage:.0%}")
    print(f"  Endpoint           : {ep_display}")
    print(f"  Properties (N/K)   : {props}")
    print(f"  Critique           : {grade.critique}")
    print()

══ Multi-query search (8 chunks) ══
  Topic matched      : ✅
  Objects coverage   : 100%
  Endpoint           : ✅ present
  Properties (N/K)   : 3 / 3
  Critique           : The retrieval successfully identified information about databases, properties like 'Due Date' and 'Select', and the process of creating pages within a database. However, the chunks are very fragmented and lack a cohesive, step-by-step guide to accomplishing the user's request.

══ CoT search (21 chunks) ══
  Topic matched      : ❌
  Objects coverage   : 33%
  Endpoint           : ✅ present
  Properties (N/K)   : 2 / 3
  Critique           : The retrieval identified some relevant API endpoints, indicating awareness of the task's need for API interaction. However, the chunks largely focus on general API details and database property configurations, lacking specific guidance on combining these elements to create a task with a due date and select property.



In [ ]:
summary = await summarize_retrieval_results(consolidated)
print(summary)


Here's a concise bullet point summary of the provided text, focusing on the Notion API and its capabilities:

* **API Focus:** The document details the Notion API, specifically the `/v1/pages` endpoint for creating new pages.
* **Parenting:** Pages can be created under existing pages or databases (using `page_id` or `database_id` in the `parent` parameter). Bots with public integrations can also create private workspace-level pages.
* **Properties:**
    *  If parent is a page, only `title` is a valid property.
    *  If parent is a database, properties must match the database's defined properties.
* **Content:** Pages can be created with or without content using the `children` option (requires content capabilities).
* **Capabilities:**  Updating content requires specific "update content" capabilities enabled in your integration settings.
* **Error Handling:**  Standard error codes apply; see the Notion API documentation for details.
* **Database Relations:** To access properties from 

#### LOOP

In [ ]:
separator = "=" * 60
max_trials = 2

# ── Step 1.5: Requirements Analysis ──────────────────────────
print(f"\n{separator}\nSTEP 1.5 — Requirements analysis …")
requirements = await analyze_requirements(user_prompt)
print(f"  {len(requirements)} critical requirements identified")


STEP 1.5 — Requirements analysis …
  5 critical requirements identified

STEP 2 — RAG retrieval (targeted) …

STEP 3 — Reflection …
*   **Define Function:** `add_task(task_name, notion_api_key, database_id)` accepting task name (string), Notion API key (string), and database ID (string).
*   **Import `requests`:** Import the necessary library for making HTTP requests.
*   **Construct Headers:** Create a dictionary for headers, including `Authorization: Bearer <notion_api_key>` and `Content-Type: application/json`.
*   **Build Request Body:** Create a dictionary representing the new task properties: `{"Name": {"title": [{"text": {"content": task_name}}]}, "Importance": {"number": 4}, "Due Date": {"date": {"start": "2026-02-22"}}}`.
*   **Define Endpoint URL:** Construct the API endpoint URL: `https://api.notion.com/v1/databases/{database_id}/pages`.
*   **Make API Request:** Use `requests.post()` to send a POST request to the endpoint with headers and the request body (converted to JSO

In [ ]:
# ── Step 2: RAG (targeted with requirements) ─────────────────
print(f"\n{separator}\nSTEP 2 — RAG retrieval (targeted) …")
if rag_data is None:
    rag_data = await rag_retrieve(user_prompt, corpora_text, requirements=requirements)


STEP 2 — RAG retrieval (targeted) …


In [ ]:
# ── Step 2: Plan + general_info ──────────────────────────────
print(f"\n{separator}\nSTEP 2 — Generating request plan …")
request_plan = await generate_request_plan(user_prompt, rag_data)
general_info = build_general_info(user_prompt, rag_data, request_plan)
print(request_plan)


In [ ]:
# ── Step 4: Generate tests (done ONCE, reused each trial) ────
print(f"\n{separator}\nSTEP 4 — Generating tests …")
test_code = await generate_tests(general_info)
print("Tests are generated. ")
write_tests(test_code)



STEP 4 — Generating tests …
Tests written → current_tests.py


In [ ]:
feedback: Optional[str] = None

for trial in range(1, max_trials + 1):
    print(f"\n{separator}\nTRIAL {trial}/{max_trials}")

    # ── Step 5: Generate code ─────────────────────────────────
    print("STEP 5 — Generating solution code …")
    code_result  = await generate_code(general_info, test_code, feedback=feedback)
    generated_code = code_result.get("code", "")
    function_name  = code_result.get("function_name", "")
    write_solution(generated_code)
    print(f"  function: {function_name}")

    # ── Step 6: Run tests ─────────────────────────────────────
    print("STEP 6 — Running pytest …")
    test_results = run_tests()
    print(f"  tests passed: {test_results['passed']}")
    if not test_results["passed"]:
        print(test_results["stdout"][-1500:])

    # ── Step 6: Judge ─────────────────────────────────────────
    print("STEP 6 — Judging code …")
    verdict = await reflect_code(general_info, generated_code, test_results)

    passed   = verdict.get("pass", False)
    feedback = verdict.get("feedback", "")

    print(f"  Verdict: {'✓ PASS' if passed else '✗ FAIL'}")

    if passed:
        print(f"\n{separator}\nAGENT COMPLETE — solution saved to solution.py")
        print({"pass": True, "trial": trial, "code": generated_code, "verdict": verdict})

    # Append feedback to general_info for next trial
    if trial < max_trials:
        print(f"\n  Feedback: {feedback}\n  → Retrying …")
        general_info = (
            general_info.rstrip("</general_info>").rstrip()
            + f"\n\n<previous_attempt_feedback trial='{trial}'>\n{feedback}\n"
                "</previous_attempt_feedback>\n</general_info>"
        )

# Exhausted trials
print(f"\n{separator}\nAGENT EXHAUSTED — max trials reached. Last feedback:\n{feedback}")
print({"pass": False, "trial": max_trials, "code": generated_code, "verdict": verdict})


TRIAL 1/2
STEP 5 — Generating solution code …
Solution written → solution.py
  function: add_task
STEP 6 — Running pytest …
  tests passed: False
31mE     Actual: post('https://api.notion.com/v1/databases/test-database-id/pages', headers={'Authorization': 'Bearer test-token', 'Content-Type': 'application/json'}, json={'parent': {'database_id': 'test-database-id'}, 'properties': {'Name': {'title': [{'text': {'content': 'Test Task'}}]}, 'Importance': {'number': 4}, 'Due Date': {'date': {'start': '2026-02-22'}}}})
___________________________ test_add_task_api_error ___________________________
current_tests.py:47: in test_add_task_api_error
    assert "Test Error" in str(excinfo.value)
E   assert 'Test Error' in "Failed to parse API response: 'id'"
E    +  where "Failed to parse API response: 'id'" = str(Exception("Failed to parse API response: 'id'"))
E    +    where Exception("Failed to parse API response: 'id'") = <ExceptionInfo Exception("Failed to parse API response: 'id'") tblen=2>.

In [ ]:
# ── Run the full agent ────────────────────────────────────────────────────────
result = await run_agent(
    user_prompt=user_prompt,
    corpora_text=corpora_text,
    notion_keys=notion_keys,
    rag_data=rag_data,  # Reuse RAG results if already computed
)

print("\nFinal result:")
print(f"  passed  : {result['pass']}")
print(f"  trial   : {result['trial']}")
print(f"  feedback: {result['verdict'].get('feedback', '')[:300]}")


STEP 2 — RAG retrieval …

STEP 3 — Reflection …
*   **Define Function Signature:** `add_task(api_key, database_id, task_name, due_date=None, assignee=None, status="To Do")`
*   **Import `requests`:** For making HTTP requests to the Notion API.
*   **Set Headers:** `headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json", "Notion-Version": "2022-06-28"}`
*   **Construct Request Body:** Create a dictionary representing the new task's properties.  Use `{"object": "page", "parent": {"database_id": database_id}, "properties": {"Task Name": {"title": [{"text": {"content": task_name}}]}, "Due Date": {"date": {"start": due_date}}, "Assignee": {"people": [{"object": "user", "id": assignee}]}, "Status": {"select": {"name": status}} }}`.
*   **API Endpoint:** Use `https://api.notion.com/v1/pages` for creating a new page.
*   **Make POST Request:** `response = requests.post(url, headers=headers, json=request_body)`
*   **Error Handling:** Check `response.status_code`. 

CancelledError: 